In [10]:
import numpy as np
import pandas as pd
from pathlib import Path
import re

In [11]:
def numericSortKey(path: Path) -> any:
    """Extracts leading integer from a folder name for numeric sorting."""
    match = re.match(r'(\d+)', path.name)
    return int(match.group(1)) if match else float('inf')


In [12]:
data_folder = './distances/vietnam/'
pop_folder = data_folder + 'facebook/'

In [13]:
current_hospitals = pd.read_pickle(data_folder+'current_hospitals.pkl')
pop = pd.read_pickle(pop_folder+'population.pkl')

In [14]:
f'{len(pop):,} {len(current_hospitals):,}'

'406,784 80'

In [15]:
stats = []

subFolders = sorted(
    [entry for entry in Path(pop_folder).iterdir() if entry.is_dir()],
    key=numericSortKey,
    reverse=True
)

for casus in subFolders:
    try:
        allHospitals = pd.read_pickle((casus / 'all_hospitals').with_suffix('.pkl'))
        newHospitals = pd.read_pickle((casus / 'new_hospitals').with_suffix('.pkl'))
        distances = pd.read_pickle((casus / 'distances_osm_max_300km').with_suffix('.pkl'))

        stats.append({
            'casus': casus.name,
            'nAllHospitals': len(allHospitals),
            'nNewHospitals': len(newHospitals),
            'nPopInDistanceData': distances.pop_id.nunique(),
            'nHospitalsInDistanceData': distances.hosp_id.nunique()
        })
    except FileNotFoundError as e:
        print(f'Skipped {casus.name}: missing file ({e})')
    except Exception as e:
        print(f'Error in {casus.name}: {e}')

pd.DataFrame(stats).set_index('casus').to_clipboard()
